In [1]:
import pandas as pd

In [2]:
df_rera = pd.read_parquet("../data/processed/clean_rer_a.parquet")
df_rerb = pd.read_parquet("../data/processed/clean_rer_b.parquet")
df_rerd = pd.read_parquet("../data/processed/clean_rer_d.parquet")
df_incidents = pd.read_parquet("../data/processed/clean_incidents.parquet")

df_all = pd.concat([df_rera, df_rerb, df_rerd], ignore_index=True)
df_valide = df_all[~df_all["est_supprime"]]

In [3]:
# Axe 2 : contagion dans le tunnel (correlation B / D, lag, resilience)

chat = df_valide[df_valide["stop_name"] == "Châtelet-Les Halles"].copy()
chat["fenetre"] = chat["aimed_time"].dt.floor("15min")
serie_chat = chat.groupby(["fenetre", "line"])["retard_min"].mean().unstack()
serie_chat.corr()

line,rer_a,rer_b,rer_d
line,,,
rer_a,1.000000,0.072867,0.135190
rer_b,0.072867,1.000000,0.259524
rer_d,0.135190,0.259524,1.000000


In [ ]:
# Decalage temporel (.shift()) a Chatelet-Les Halles par pas de 15 min
# Grille reguliere de 15 min (asfreq) pour ne pas sauter la coupure nocturne.
# La correlation B vers D culmine a 0,271 au lag +15 min (0,260 a 0 min),
# contre 0,07 a 0,11 pour le placebo B vers A.

grille = serie_chat.asfreq("15min")

lags = []
for k in range(-4, 5):  # k > 0 : B a t compare a l'autre ligne a t + 15*k min
    lags.append({
        "lag_min": k * 15,
        "B_vers_D": grille["rer_b"].corr(grille["rer_d"].shift(-k)),
        "B_vers_A": grille["rer_b"].corr(grille["rer_a"].shift(-k)),
        "D_vers_A": grille["rer_d"].corr(grille["rer_a"].shift(-k)),
    })

df_lags = pd.DataFrame(lags).set_index("lag_min").round(3)
df_lags

In [ ]:
# Correlation B / D par sens (Nord vs Sud) dans le tunnel
# Separation des deux voies du tunnel (Chatelet-Les Halles et Gare du Nord).
# La correlation est plus marquee en sens Sud (0,288 a lag 0 et 0,294 a lag +15 min
# a Chatelet, 0,255 a Gare du Nord) qu'en sens Nord (0,12 a 0,15).

bd_tunnel = df_valide[
    df_valide["line"].isin(["rer_b", "rer_d"])
    & df_valide["zone_tunnel"]
    & df_valide["direction"].isin(["Nord", "Sud"])
].copy()
bd_tunnel["fenetre"] = bd_tunnel["aimed_time"].dt.floor("15min")

lignes_sens = []
for gare in ["Châtelet-Les Halles", "Gare du Nord"]:
    for sens in ["Nord", "Sud"]:
        sub = bd_tunnel[(bd_tunnel["stop_name"] == gare) & (bd_tunnel["direction"] == sens)]
        g = sub.groupby(["fenetre", "line"])["retard_min"].mean().unstack().asfreq("15min")
        lignes_sens.append({
            "gare": gare,
            "direction": sens,
            "lag_-15": g["rer_b"].corr(g["rer_d"].shift(1)),
            "lag_0": g["rer_b"].corr(g["rer_d"]),
            "lag_+15": g["rer_b"].corr(g["rer_d"].shift(-1)),
        })

corr_sens = pd.DataFrame(lignes_sens).set_index(["gare", "direction"]).round(3)
corr_sens

In [ ]:
# Effet conditionnel et test placebo RER A a Chatelet-Les Halles
# Quand B depasse 10 min de retard moyen, le retard moyen de D passe de 2,27 a 7,47 min (x3,3)
# alors que le RER A dans la meme gare reste a 1,16 min.

etat_b = pd.cut(
    grille["rer_b"],
    bins=[-15, 3, 10, 180],
    labels=["B fluide (<3 min)", "B ralenti (3-10 min)", "B perturbe (>10 min)"],
)
grille.groupby(etat_b, observed=False)[["rer_b", "rer_d", "rer_a"]].agg(["count", "mean", "median"]).round(2)

In [ ]:
# Resilience : temps de retour sous 3 min apres un pic > 10 min
# Mesure la duree pour repasser sous 3 min sans traverser la coupure nocturne (> 30 min sans train).
# Sur le troncon central, B met 194 min en moyenne (mediane 150 min, P90 432 min) et D 105 min
# (P90 210 min) contre 81 min (P90 133,5 min) sur le A, avec 3 a 4 fois plus d'episodes sur B et D.

def calcul_resilience(grille_df, seuil_pic=10, seuil_retour=3, pas_min=15):
    resultats = []
    for col in ["rer_a", "rer_b", "rer_d"]:
        vals = grille_df[col].values
        durees = []
        i = 0
        n = len(vals)
        while i < n:
            if pd.notna(vals[i]) and vals[i] > seuil_pic:
                j = i + 1
                nan_streak = 0
                while j < n:
                    if pd.isna(vals[j]):
                        nan_streak += 1
                        if nan_streak > 2:
                            break
                    else:
                        nan_streak = 0
                        if vals[j] < seuil_retour:
                            durees.append((j - i) * pas_min)
                            break
                    j += 1
                i = j + 1
            else:
                i += 1
        ds = pd.Series(durees, dtype=float)
        resultats.append({
            "line": col,
            "nb_episodes": len(ds),
            "mean_min": ds.mean(),
            "median_min": ds.median(),
            "p90_min": ds.quantile(0.90),
        })
    return pd.DataFrame(resultats).set_index("line")


tmp = df_valide.copy()
tmp["fenetre"] = tmp["aimed_time"].dt.floor("15min")
grille_troncon = tmp.groupby(["fenetre", "line"])["retard_min"].mean().unstack().asfreq("15min")

res_chatelet = calcul_resilience(grille)
res_troncon = calcul_resilience(grille_troncon)
pd.concat({"Châtelet-Les Halles": res_chatelet, "Tronçon central": res_troncon}).round(1)